In [0]:
%sql

USE CATALOG `retail-dwh-project`;

CREATE SCHEMA IF NOT EXISTS clean;

-- =========================
-- CUSTOMERS CLEAN
-- =========================

CREATE OR REPLACE TABLE clean.customers_clean AS
SELECT DISTINCT
    CustomerID,
    INITCAP(TRIM(CustomerName)) AS CustomerName,
    LOWER(TRIM(Email))          AS Email,
    TRIM(City)                  AS City,
    TRIM(Address)               AS Address,
    LastUpdated
FROM bronze.customers_raw;

-- =========================
-- PRODUCTS CLEAN
-- =========================

CREATE OR REPLACE TABLE clean.products_clean AS
SELECT DISTINCT
    ProductID,
    TRIM(ProductName) AS ProductName,
    TRIM(Category)    AS Category,
    UnitPrice
FROM bronze.products_raw
WHERE UnitPrice > 0;

-- =========================
-- STORES CLEAN
-- =========================

CREATE OR REPLACE TABLE clean.stores_clean AS
SELECT DISTINCT
    StoreID,
    INITCAP(TRIM(StoreName))          AS StoreName,
    COALESCE(TRIM(Region), 'Unknown') AS Region
FROM bronze.stores_raw;

-- =========================
-- SALES CLEAN
-- =========================

CREATE OR REPLACE TABLE clean.sales_clean AS
SELECT DISTINCT
    TransactionID,
    CustomerID,
    ProductID,
    StoreID,
    Quantity,
    TO_DATE(TxnDate, 'dd-MM-yyyy') AS TxnDate
FROM bronze.sales_raw
WHERE Quantity > 0
  AND CustomerID IN (
      SELECT CustomerID
      FROM clean.customers_clean
  );

-- =========================
-- COUNT CLEAN TABLES
-- =========================

SELECT 'customers_clean' AS table_name, COUNT(*) AS total_rows
FROM clean.customers_clean

UNION ALL

SELECT 'products_clean', COUNT(*)
FROM clean.products_clean

UNION ALL

SELECT 'stores_clean', COUNT(*)
FROM clean.stores_clean

UNION ALL

SELECT 'sales_clean', COUNT(*)
FROM clean.sales_clean;